# 可求导函数

## 理解可求导函数

- 可求导函数主要提供记录操作历史并定义用于区分操作公式的一种框架设计。
    - 每个Tensor的操作都产生一个新的函数对象，函数对象主要完成如下功能：
        1. 记录保存（使用DAG格式保存，一种动态的计算图）；
            - 这种图的最大好处是，在求导的时候，可以你想根据拓扑结构，调用backward，层层传递导数（这种思维来自链式求导的数学原理）
        2. 执行计算；

- 备注：
    - 在PyTorch中，计算路径通过动态计算图(Dynamic Computational Graph, DCG)描述。DGC由变量和函数组成，包含两层含义：
        1. 计算图
            - 通过图的方式描述数学计算过程。在计算图中，节点由输入和计算函数组成，数据通过边流向下一个节点。PyTorch的计算图是有向无环图(Directed Acyclic Graph： **DAG** )，叶子节点由输入变量组成，根节点由输出变量组成，中间节点既可以是算术操作符，也可以中间结果变量。
        2. 动态图
            - 在Caffe以及早期版本的TensorFlow中，计算图是静态的，即预先定义好计算的顺序。运行过程中只是数据的流通，计算图是不可变动的。在PyTorch中计算图是动态的，边构建边定义(define-by-run)。动态图支持构建过程中随时查看中间结果，而静态图则需要定义图之后，运行整个图才能查看结果。此外，动态图天然支持运行过程中结构动态变化的神经网络架构。当然，由于每次迭代都会构建一个全新的计算图，会影响计算速度。

## Function类实现说明

- forward提供函数的输出计算
- backward提供用户自己的求导实现

- 下面是使用模式与执行流程。

In [29]:
import torch 
from torch.autograd import Function
 
# 定义可自动求导函数类
class MyFunction(Function):
    @staticmethod
    def forward(ctx, w, x, b):
        # 上下文存储
        ctx.save_for_backward(w, x) 
        # 计算输出（张量）
        output = w * x + b
        print("forward输出", output)
        return output
    
    @staticmethod
    def backward(ctx, grad_output):
        print("backward输入：", grad_output)
        # 上下文数据获取
        w, x = ctx.saved_tensors
        # 下面返回求出的导数（下面使用的是不正确的导数求法，只是说明执行过程）
        return torch.Tensor([77, 77, 77]), torch.Tensor([88, 88, 88]), torch.Tensor([99, 99, 99])



x = torch.Tensor([1, 2, 3])
w = torch.Tensor([2, 3, 4])
w.requires_grad=True
b = torch.Tensor([3, 4, 5])
b.requires_grad=True

print('>>>>>>>>>>>>>>>>>>>')
# z=MyFunction.apply(w, x, b)                # 调用的就是forward函数
z=MyFunction.apply(w, x, b)                           # 使用可调用方式
print('<<<<<<<<<<<<<<<<<<<')
z.backward(torch.Tensor([1, 1, 1]))
 
# x不需要求导，中间过程还是会计算它的导数，但随后被清空
print(x.grad, w.grad, b.grad)
print(z.grad_fn)  
print(help(z.grad_fn))

>>>>>>>>>>>>>>>>>>>
forward输出 tensor([ 5., 10., 17.])
<<<<<<<<<<<<<<<<<<<
backward输入： tensor([1., 1., 1.])
None tensor([77., 77., 77.]) tensor([99., 99., 99.])
Help on MyFunctionBackward in module torch.autograd.function object:

class MyFunctionBackward(BackwardCFunction)
 |  Method resolution order:
 |      MyFunctionBackward
 |      BackwardCFunction
 |      torch._C._FunctionBase
 |      _ContextMethodMixin
 |      _HookMixin
 |      builtins.object
 |  
 |  Methods inherited from BackwardCFunction:
 |  
 |  apply(self, *args)
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors inherited from BackwardCFunction:
 |  
 |  __dict__
 |      dictionary for instance variables (if defined)
 |  
 |  __weakref__
 |      list of weak references to the object (if defined)
 |  
 |  ----------------------------------------------------------------------
 |  Methods inherited from torch._C._FunctionBase:
 |  
 |  __new__(*args, **kwargs) from built

2. BackwardCFunction的apply使用，等价于backward的调用

In [35]:
import torch 
from torch.autograd import Function
 
# 定义可自动求导函数类
class MyFunction(Function):
    @staticmethod
    def forward(ctx, w, x, b):
        # 上下文存储
        ctx.save_for_backward(w, x) 
        # 计算输出（张量）
        output = w * x + b
        print("forward输出", output)
        return output
    
    @staticmethod
    def backward(ctx, grad_output):
        print("backward输入：", grad_output)
        # 上下文数据获取
        w, x = ctx.saved_tensors
        # 下面返回求出的导数（下面使用的是不正确的导数求法，只是说明执行过程）
        return torch.Tensor([77, 77, 77]), torch.Tensor([88, 88, 88]), torch.Tensor([99, 99, 99])



x = torch.Tensor([1, 2, 3])
w = torch.Tensor([2, 3, 4])
w.requires_grad=True
b = torch.Tensor([3, 4, 5])
b.requires_grad=True

print('>>>>>>>>>>>>>>>>>>>')
z=MyFunction.apply(w, x, b)                # 调用的就是forward函数
# z=MyFunction()(w, x, b)                 # 非静态的forward与backward已经不推荐使用
print(z)     # 这个是张量Tensor，就是forward的输出
print('<<<<<<<<<<<<<<<<<<<')

re = z.grad_fn.apply(torch.Tensor([1, 1, 1]))
 
# x不需要求导，中间过程还是会计算它的导数，但随后被清空
print(re)   # 直接输出
print(x.grad, w.grad, b.grad)    # 都为None
print(z.grad_fn)  

>>>>>>>>>>>>>>>>>>>
forward输出 tensor([ 5., 10., 17.])
tensor([ 5., 10., 17.], grad_fn=<MyFunctionBackward>)
<<<<<<<<<<<<<<<<<<<
backward输入： tensor([1., 1., 1.])
(tensor([77., 77., 77.]), tensor([88., 88., 88.]), tensor([99., 99., 99.]))
None None None


## 实现Sigmoid求导的例子

- forward的计算比较简单，公式表示如下
    - $s(x) = \dfrac{1}{1 + e ^{-x}}$
- backward利用S函数的导数函数直接计算，S函数的导数可以使用自己表示；公式表示如下：
    - $s^{\prime}(x)=s(x)(1-s(x))$

In [38]:
import torch 
from torch.autograd import Function
 
# 定义可自动求导函数类
class SigmoidFun(Function):
    @staticmethod
    def forward(ctx, x):   # S函数的参数只有一个
        output = 1.0 / (1 + (-x).exp()) 
        # 下面要使用output，所以存储
        ctx.save_for_backward(output)
        return output
    
    @staticmethod
    def backward(ctx, grad_output):   # grad_output一定与x同型，可以用来做内积运算
        # 取出上面计算的output，然后根据已有的导数公式计算导数
        out, = ctx.saved_tensors    # 返回的是元组
        grad = out * (1 - out) * grad_output
        return grad    # 返回的张量必须与forwad的输入参数对应


x = torch.Tensor([1, 2, 3])
x.requires_grad=True
y = SigmoidFun.apply(x) 
y.backward(torch.Tensor([1, 1, 1]))
print(x.grad)

tensor([0.1966, 0.1050, 0.0452])


---- 

- 对损失函数来说，backward的计算方式可以根据误差项来计算。

# 梯度检查

- 上面的导数计算是否正确？或者误差能否忍受？Torch提供了梯度检查，检查的原理是导数的极限定义，采用无穷小$\epsilon$，来模拟导数的逼近。

## 函数定义

```python
    gradcheck(
        func,       # 检测的函数，输入参数是张量序列，返回的是张量或者张量元组，需要是可调用函数，需要使用函数类的.apply成员或者函数的调用形式。
        inputs,     # 输入张量序列
        eps=1e-06,     # 用来计算近似导数的无穷小量（perturbation for finite differences）
        atol=1e-05,     # 可以接受的绝对误差
        rtol=0.001,      # 可以接受的相对误差
        raise_exception=True,   # 如果不在误差范围内，则抛出异常
        check_sparse_nnz=False,     # 是否检测稀疏张量，nnz表示只对非零位置检测（nnz:Number of  Non-Zero）
        nondet_tol=0.0)   # 对相同输入的结果误差的忍受度。一般都是0
```

- 返回值：
    - 如果导数计算没有问题，则返回True

## 误差检测的原理

- 因为导数的定义如下：
    - $f^{\prime}(x) = \dfrac{\partial{f}}{\partial{x}} = \lim \limits _{\epsilon \to 0} \dfrac{f(x+\epsilon) - f(x - \epsilon)}{2\epsilon}$
    
    - 实际上导数的计算只要取足够小的无穷小量，就可以用来替代导数。

## 梯度检查的使用例子

In [59]:
import torch 
from torch.autograd import Function
 
# 定义可自动求导函数类
class SigmoidFun(Function):
    @staticmethod
    def forward(ctx, x):   # S函数的参数只有一个
        output = 1.0 / (1 + (-x).exp()) 
        # 下面要使用output，所以存储
        ctx.save_for_backward(output)
        return output
    
    @staticmethod
    def backward(ctx, grad_output):   # grad_output一定与x同型，可以用来做内积运算
        # 取出上面计算的output，然后根据已有的导数公式计算导数
        out, = ctx.saved_tensors    # 返回的是元组
        grad = out * (1 - out) * grad_output
        grad += 0.0000000000000000000000000000000001     # 故意产生点误差
        return grad    # 返回的张量必须与forwad的输入参数对应


x = torch.Tensor([1, 2, 3])
x.requires_grad=True
y = SigmoidFun.apply(x) 
y.backward(torch.Tensor([1, 1, 1]))     # 函数的输出就可以实现自动求导了（实际上导数是用户自己定义实现的）
print(x.grad)

# 梯度检测
x_input = torch.DoubleTensor([1, 2, 3])    # 检测梯度的输入，需要满足双精度的条件
x_input.requires_grad=True
b = torch.autograd.gradcheck(SigmoidFun.apply,  x_input,  eps=1e-8, raise_exception=False)
print("检测结果", b)

tensor([0.1966, 0.1050, 0.0452])
检测结果 False


- 注意：
    - 上面的导数，使用Tensor的运算也可以计算导数，但是使用内置运算的求导因为对过程跟踪比较消耗资源，所以可以测试我们自己实现的求导应该快不少，
    - 使用torch的内置sigmoid函数(torch.sigmoid)实现自动求导，计算速度应该是最快的！

----